# Анализ городской среды: BlocksNet 1.0.0a9

**Территория:** Санкт-Петербург, Петроградский район  
**Данные:** OpenStreetMap + готовые графы и данные проекта blocksnet-data (v0.0.1)

Ноутбук повторяет официальный pipeline BlocksNet и выполняет полный цикл анализа кварталов:

1. нарезка кварталов из данных OSM (дороги, железные дороги, вода);
2. назначение землепользования (land use);
3. **STOP** — проверка промежуточного результата;
4. загрузка готовых графов (автомобильный и интермодальный);
5. расчёт матриц транспортной доступности (время в пути, минуты);
6. здания и население с агрегацией по кварталам;
7. школы и обеспеченность (competitive provision);
8. показатели плотности (FSI, GSI, MXI, L, OSR);
9. морфотипы (Spacematrix, Strelka);
10. доступность по площади (area accessibility);
11. сводная таблица показателей `blocks_metrics.geojson` — основа для карты и дашборда (этап 2).

## Подготовка окружения (Python 3.12)

BlocksNet 1.0.0a9 требует Python < 3.13. В корне репозитория выполните:

    py -3.12 -m venv .venv312
    .venv312\Scripts\pip install -e ".[dev]"
    .venv312\Scripts\python -m ipykernel install --user --name blocksnet312 --display-name "Python 3.12 (blocksnet)"

Выберите ядро **Python 3.12 (blocksnet)** и запускайте ноутбук из папки `examples/`. Все промежуточные результаты кэшируются в `examples/data/blocksnet/` и переиспользуются при повторных запусках.


In [ ]:
import pickle
import shutil
import zipfile
from enum import Enum
from pathlib import Path

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import requests

from blocksnet.config import service_types_config

DATA_DIR = Path.cwd() / 'data' / 'blocksnet'
DATA_DIR.mkdir(parents=True, exist_ok=True)

TERRITORY = 'Санкт-Петербург, Петроградский район'
GRAPH_ARCHIVE_URL = 'https://github.com/IDUclub/blocksnet-data/releases/download/v0.0.1/saint_petersburg.zip'
SERVICE_TYPE = 'school'
MATRIX_FOR_ANALYSIS = 'intermodal'

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 220)


def read_cache(filename):
    path = DATA_DIR / filename
    if not path.exists():
        return None
    gdf = gpd.read_file(path)
    if 'block_id' in gdf.columns:
        gdf = gdf.set_index('block_id')
        gdf.index = gdf.index.astype(int)
        gdf = gdf.sort_index()
    print(f'читаем из кэша: {filename} ({len(gdf)} объектов)')
    return gdf


def _flatten(value):
    if isinstance(value, Enum):
        return value.value
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, (list, tuple, set, dict)):
        return str(value)
    return value


def write_cache(gdf, filename):
    flat = gdf.copy()
    for column in flat.columns:
        if column != flat.geometry.name:
            flat[column] = flat[column].map(_flatten)
    flat.to_crs(4326).to_file(DATA_DIR / filename, driver='GeoJSON')
    print(f'сохранили в кэш: {filename} ({len(flat)} объектов)')


def read_matrix(filename):
    path = DATA_DIR / filename
    if not path.exists():
        return None
    matrix = pd.read_csv(path, index_col=0)
    matrix.index = matrix.index.astype(int)
    matrix.columns = matrix.columns.astype(int)
    print(f'читаем из кэша: {filename} {matrix.shape}')
    return matrix


def write_matrix(matrix, filename):
    matrix.to_csv(DATA_DIR / filename)
    print(f'сохранили в кэш: {filename} {matrix.shape}')


def plot_gdf(gdf, column=None, **kwargs):
    ax = gdf.plot(column=column, **kwargs)
    ax.set_axis_off()
    return ax


## 1. Нарезка кварталов

### 1.1. Граница территории

Граница района берётся из OSM по названию (Osmnx / Nominatim).


In [ ]:
boundaries = read_cache('boundary.geojson')
if boundaries is None:
    boundaries = ox.geocode_to_gdf(TERRITORY)
    write_cache(boundaries, 'boundary.geojson')

crs = boundaries.estimate_utm_crs()
boundaries = boundaries.to_crs(crs)

plot_gdf(boundaries)
print(f'Аналитическая CRS: {crs.name}')


### 1.2. Слои OSM

Скачиваем воду, дороги и железные дороги в границах района. Набор тегов — как в официальном pipeline BlocksNet.


In [ ]:
bc_tags = {
    'roads': {
        'highway': ['construction', 'crossing', 'living_street', 'motorway', 'motorway_link', 'motorway_junction', 'pedestrian', 'primary', 'primary_link', 'raceway', 'residential', 'road', 'secondary', 'secondary_link', 'services', 'tertiary', 'tertiary_link', 'track', 'trunk', 'trunk_link', 'turning_circle', 'turning_loop', 'unclassified'],
        'service': ['living_street', 'emergency_access'],
    },
    'railways': {
        'railway': 'rail',
    },
    'water': {
        'riverbank': True,
        'reservoir': True,
        'basin': True,
        'dock': True,
        'canal': True,
        'pond': True,
        'natural': ['water', 'bay'],
        'waterway': ['river', 'canal', 'ditch'],
        'landuse': 'basin',
        'water': 'lake',
    },
}

boundary_polygon = boundaries.to_crs(4326).union_all()

water = read_cache('water.geojson')
if water is None:
    water = ox.features_from_polygon(boundary_polygon, bc_tags['water'])
    water = water[water.geom_type.isin(['Polygon', 'MultiPolygon', 'LineString', 'MultiLineString'])].copy()
    write_cache(water, 'water.geojson')
water = water.to_crs(crs)

roads = read_cache('roads.geojson')
if roads is None:
    roads = ox.features_from_polygon(boundary_polygon, bc_tags['roads'])
    roads = roads[roads.geom_type.isin(['LineString', 'MultiLineString'])].copy()
    write_cache(roads, 'roads.geojson')
roads = roads.to_crs(crs)

railways = read_cache('railways.geojson')
if railways is None:
    railways = ox.features_from_polygon(boundary_polygon, bc_tags['railways'])
    railways = railways[railways.geom_type.isin(['LineString', 'MultiLineString'])].copy()
    write_cache(railways, 'railways.geojson')
railways = railways.to_crs(crs)

print(f'вода: {len(water)}, дороги: {len(roads)}, железные дороги: {len(railways)}')


### 1.3. Нарезка кварталов

Кварталы нарезаются линиями дорог и железных дорог, полигоны воды исключаются из кварталов.


In [ ]:
from blocksnet.blocks.cutting import cut_urban_blocks, preprocess_urban_objects

blocks = read_cache('blocks_raw.geojson')
if blocks is None:
    lines, polygons = preprocess_urban_objects(roads, railways, water)
    blocks = cut_urban_blocks(boundaries, lines, polygons)
    write_cache(blocks.assign(block_id=blocks.index), 'blocks_raw.geojson')
blocks = blocks.to_crs(crs)

plot_gdf(blocks, color='#dddddd')
print(f'Кварталов нарезано: {len(blocks)}')


## 2. Землепользование

### 2.1. Функциональные зоны OSM

Функциональные зоны (landuse) скачиваются из OSM; правила маппинга в `LandUse` — как в официальном pipeline.


In [ ]:
functional_zones = read_cache('functional_zones.geojson')
if functional_zones is None:
    functional_zones = ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags={'landuse': True})
    functional_zones = functional_zones.reset_index(drop=True)[['geometry', 'landuse']].rename(columns={'landuse': 'functional_zone'})
    write_cache(functional_zones, 'functional_zones.geojson')
functional_zones = functional_zones.to_crs(crs)

functional_zones.head()


In [ ]:
from blocksnet.blocks.assignment import assign_land_use
from blocksnet.enums import LandUse

rules = {
    'commercial': LandUse.BUSINESS,
    'industrial': LandUse.INDUSTRIAL,
    'cemetery': LandUse.SPECIAL,
    'garages': LandUse.INDUSTRIAL,
    'residential': LandUse.RESIDENTIAL,
    'retail': LandUse.BUSINESS,
    'grass': LandUse.RECREATION,
    'farmland': LandUse.AGRICULTURE,
    'construction': LandUse.SPECIAL,
    'brownfield': LandUse.INDUSTRIAL,
    'forest': LandUse.RECREATION,
    'recreation_ground': LandUse.RECREATION,
    'religious': LandUse.SPECIAL,
    'flowerbed': LandUse.RECREATION,
    'military': LandUse.SPECIAL,
    'landfill': LandUse.TRANSPORT,
}

blocks_land_use = read_cache('blocks_land_use.geojson')
if blocks_land_use is None:
    blocks = assign_land_use(blocks, functional_zones, rules)
    blocks_land_use = blocks
    write_cache(blocks_land_use.assign(block_id=blocks_land_use.index), 'blocks_land_use.geojson')
blocks = blocks_land_use.to_crs(crs)

ax = blocks.plot(color='#dddddd')
blocks.plot(column='land_use', legend=True, ax=ax).set_axis_off()
print(blocks.land_use.value_counts(dropna=False).to_string())


---

## STOP: проверка промежуточного результата

Кварталы нарезаны, землепользование назначено. Данные лежат в кэше `examples/data/blocksnet/`:

- `blocks_raw.geojson` — нарезанные кварталы;
- `blocks_land_use.geojson` — кварталы с землепользованием.

**Посмотрите на карту выше.** Дальше по умолчанию считаются: матрицы доступности (авто и ОТ), население, обеспеченность школами, плотность, морфотипы и доступность по площади. Набор можно изменить в первой ячейке: `SERVICE_TYPE` (тип сервиса) и `MATRIX_FOR_ANALYSIS` (`drive` или `intermodal`).

Следующая ячейка показывает, какие сервисы и модули анализа доступны в BlocksNet.


In [ ]:
import pkgutil

import blocksnet.analysis as analysis_package

service_names = sorted(service_types_config)
print(f'Типов сервисов в конфиге BlocksNet: {len(service_names)}')
print(', '.join(service_names))
print()
modules = sorted(module.name for module in pkgutil.iter_modules(analysis_package.__path__))
print('Модули blocksnet.analysis: ' + ', '.join(modules))


## 3. Готовые графы транспорта

Используем открытый набор **blocksnet-data** (Санкт-Петербург, v0.0.1): он избавляет от длительной сборки графов из OSM. Вес рёбер — `time_min` (минуты).

- `graph_drive` — поездки на личном автомобиле;
- `graph_intermodal` — общественный транспорт + пешие пересадки.


In [ ]:
GRAPHS = {
    'drive': DATA_DIR / 'graph_drive.pickle',
    'intermodal': DATA_DIR / 'graph_intermodal.pickle',
}
ARCHIVE = DATA_DIR / 'saint_petersburg.zip'

if not all(path.exists() for path in GRAPHS.values()):
    if not ARCHIVE.exists():
        print('Скачиваем архив графов (~405 МБ), это может занять несколько минут...')
        with requests.get(GRAPH_ARCHIVE_URL, stream=True, timeout=120) as response:
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            loaded = 0
            with open(ARCHIVE, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    file.write(chunk)
                    loaded += len(chunk)
                    if total:
                        print(f'{loaded / 1e6:7.0f} из {total / 1e6:.0f} МБ ({100 * loaded / total:3.0f}%)')
    with zipfile.ZipFile(ARCHIVE) as archive:
        names = archive.namelist()
        for path in GRAPHS.values():
            member = next(name for name in names if name.endswith(path.name))
            with archive.open(member) as source, open(path, 'wb') as target:
                shutil.copyfileobj(source, target, length=8 * 1024 * 1024)
            print(f'извлекли из архива: {path.name}')
else:
    print('Графы уже сохранены в кэше')


In [ ]:
graphs = {}
for key, path in GRAPHS.items():
    with open(path, 'rb') as file:
        graphs[key] = pickle.load(file)

for key, graph in graphs.items():
    print(f'{key}: {type(graph).__name__}, узлов {graph.number_of_nodes()}, рёбер {graph.number_of_edges()}, crs {graph.graph.get("crs")}')

for key, graph in graphs.items():
    _, node_data = next(iter(graph.nodes(data=True)))
    assert 'x' in node_data and 'y' in node_data, f'{key}: в узлах нет координат x/y'
    edge_attributes = next(iter(graph.edges(keys=False, data=True)))[2]
    assert 'time_min' in edge_attributes, f'{key}: у рёбер нет времени поездки time_min'

print('Проверка графов пройдена: в узлах есть координаты x/y, у рёбер — время поездки time_min')


## 4. Матрицы транспортной доступности

Матрица — время в пути между центроидами всех пар кварталов (минуты). Считаем обе: на автомобиле и на транспорте.


In [ ]:
from blocksnet.relations import calculate_accessibility_matrix

acc_mx_drive = read_matrix('acc_mx_drive.csv')
if acc_mx_drive is None:
    acc_mx_drive = calculate_accessibility_matrix(blocks, graphs['drive'])
    write_matrix(acc_mx_drive, 'acc_mx_drive.csv')

acc_mx_intermodal = read_matrix('acc_mx_intermodal.csv')
if acc_mx_intermodal is None:
    acc_mx_intermodal = calculate_accessibility_matrix(blocks, graphs['intermodal'])
    write_matrix(acc_mx_intermodal, 'acc_mx_intermodal.csv')

print('Среднее время в пути между кварталами (мин):')
print(f'  на автомобиле: {acc_mx_drive.to_numpy().mean():.1f}')
print(f'  на транспорте: {acc_mx_intermodal.to_numpy().mean():.1f}')


In [ ]:
MATRICES = {'drive': acc_mx_drive, 'intermodal': acc_mx_intermodal}

for label, matrix in MATRICES.items():
    nan_count = int(matrix.isna().sum().sum())
    if nan_count:
        fill_value = float(matrix.max().max())
        print(f'{label}: {nan_count} недостижимых пар кварталов, заполняем значением {fill_value:.1f} мин')
        matrix = matrix.fillna(fill_value)
    values = matrix.to_numpy(dtype=float, copy=True)
    np.fill_diagonal(values, 0.0)
    matrix = pd.DataFrame(values, index=matrix.index, columns=matrix.columns)
    assert matrix.shape == (len(blocks), len(blocks)), label
    assert int(matrix.isna().sum().sum()) == 0, label
    MATRICES[label] = matrix

acc_mx_drive = MATRICES['drive']
acc_mx_intermodal = MATRICES['intermodal']

assert list(acc_mx_drive.index) == list(blocks.index)
print('Матрицы согласованы с кварталами:', acc_mx_drive.shape)


## 5. Здания и население

Здания скачиваются из OSM. Пропуски достраиваются модулем `impute_buildings` (норматив жилой обеспеченности 30 м²/чел.), затем данные агрегируются по кварталам.


In [ ]:
buildings = read_cache('buildings.geojson')
if buildings is None:
    buildings = ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags={'building': True}).reset_index(drop=True)
    is_living_tags = ['residential', 'house', 'apartments', 'detached', 'terrace', 'dormitory']
    buildings['is_living'] = buildings['building'].apply(lambda value: value in is_living_tags)
    buildings['number_of_floors'] = pd.to_numeric(buildings['building:levels'], errors='coerce')
    write_cache(buildings, 'buildings.geojson')
buildings = buildings.to_crs(crs)

print(f'Зданий в границах: {len(buildings)}')


In [ ]:
from blocksnet.blocks.aggregation import aggregate_objects
from blocksnet.preprocessing.imputing import impute_buildings

buildings_blocks = read_cache('buildings_blocks.geojson')
if buildings_blocks is None:
    buildings = impute_buildings(buildings, default_living_demand=30)
    buildings_blocks = aggregate_objects(blocks, buildings)[0]
    write_cache(buildings_blocks.assign(block_id=buildings_blocks.index), 'buildings_blocks.geojson')
buildings_blocks = buildings_blocks.to_crs(crs)

print(f'Население района: {buildings_blocks.population.sum():,.0f} чел.')
buildings_blocks.plot('population', legend=True).set_axis_off()


## 6. Школы и обеспеченность

Школы скачиваются из OSM (`amenity=school`), вместимость достраивается конфигом BlocksNet. Обеспеченность считается конкурентной моделью `competitive_provision` по выбранной матрице (`MATRIX_FOR_ANALYSIS`).


In [ ]:
from blocksnet.preprocessing.imputing import impute_services

schools = read_cache('schools.geojson')
if schools is None:
    schools = ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags={'amenity': SERVICE_TYPE}).reset_index(drop=True)
    schools = impute_services(schools, SERVICE_TYPE)
    write_cache(schools, 'schools.geojson')
schools = schools.to_crs(crs)

schools_blocks = read_cache('schools_blocks.geojson')
if schools_blocks is None:
    schools_blocks = aggregate_objects(blocks, schools)[0]
    write_cache(schools_blocks.assign(block_id=schools_blocks.index), 'schools_blocks.geojson')
schools_blocks = schools_blocks.to_crs(crs)

print(f'Школ: {len(schools)}, суммарная вместимость: {schools.capacity.sum():,.0f} мест')


In [ ]:
from blocksnet.analysis.provision import competitive_provision

service_config = service_types_config[SERVICE_TYPE]
demand = service_config['demand']
accessibility = service_config['accessibility']
print(f'{SERVICE_TYPE}: норматив {demand} мест на человека, порог доступности {accessibility} мин')

provision_blocks = read_cache('provision_school.geojson')
if provision_blocks is None:
    provision_input = buildings_blocks[['population']].join(schools_blocks[['capacity']])
    provision_mx = MATRICES[MATRIX_FOR_ANALYSIS]
    prov_df, links_df = competitive_provision(provision_input, provision_mx, accessibility, demand)
    provision_blocks = blocks.join(prov_df)
    write_cache(provision_blocks.assign(block_id=provision_blocks.index), 'provision_school.geojson')
provision_blocks = provision_blocks.to_crs(crs)
prov_df = provision_blocks.drop(columns='geometry')

print(f'Средняя обеспеченность (provision_strong): {provision_blocks.provision_strong.mean():.2f}')

ax = blocks.plot(color='#dddddd')
provision_blocks.plot('provision_strong', ax=ax, cmap='RdYlGn', vmin=0, vmax=1, legend=True)
ax.set_axis_off()


## 7. Плотность и морфотипы

Показатели плотности: FSI, GSI, MXI, L, OSR. На их основе строятся морфотипы Spacematrix и Strelka.


In [ ]:
from blocksnet.analysis.indicators import calculate_density_indicators

density_blocks = read_cache('density_indicators.geojson')
if density_blocks is None:
    density_input = buildings_blocks.copy()
    density_input['site_area'] = density_input.area
    density_df = calculate_density_indicators(density_input)
    density_blocks = density_input[['geometry']].join(density_df)
    write_cache(density_blocks.assign(block_id=density_blocks.index), 'density_indicators.geojson')
    density_df.to_csv(DATA_DIR / 'density_indicators.csv')
else:
    density_df = pd.read_csv(DATA_DIR / 'density_indicators.csv', index_col=0)
    density_df.index = density_df.index.astype(int)
density_blocks = density_blocks.to_crs(crs)

density_df.head()


In [ ]:
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes

spacematrix_blocks = read_cache('morphotypes_spacematrix.geojson')
if spacematrix_blocks is None:
    spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(density_df)
    spacematrix_df.to_csv(DATA_DIR / 'morphotypes_spacematrix.csv')
    spacematrix_clusters.to_csv(DATA_DIR / 'morphotypes_spacematrix_clusters.csv')
    spacematrix_blocks = blocks[['geometry']].join(spacematrix_df)
    write_cache(spacematrix_blocks.assign(block_id=spacematrix_blocks.index), 'morphotypes_spacematrix.geojson')
else:
    spacematrix_df = pd.read_csv(DATA_DIR / 'morphotypes_spacematrix.csv', index_col=0)
    spacematrix_df.index = spacematrix_df.index.astype(int)

ax = blocks.plot(color='#dddddd')
spacematrix_blocks.plot('morphotype', legend=True, ax=ax)
ax.set_axis_off()


In [ ]:
from blocksnet.analysis.morphotypes import get_strelka_morphotypes

strelka_blocks = read_cache('morphotypes_strelka.geojson')
if strelka_blocks is None:
    strelka_df = get_strelka_morphotypes(density_df)
    strelka_df.to_csv(DATA_DIR / 'morphotypes_strelka.csv')
    strelka_blocks = blocks[['geometry']].join(strelka_df)
    write_cache(strelka_blocks.assign(block_id=strelka_blocks.index), 'morphotypes_strelka.geojson')
else:
    strelka_df = pd.read_csv(DATA_DIR / 'morphotypes_strelka.csv', index_col=0)
    strelka_df.index = strelka_df.index.astype(int)

ax = blocks.plot(color='#dddddd')
strelka_blocks.plot('morphotype', legend=True, ax=ax)
ax.set_axis_off()


## 8. Доступность по площади

`area_accessibility` — средневзвешенное по площади зданий время доступности квартала (минуты): чем меньше, тем лучше. Считаем для обеих матриц — автомобильной и интермодальной.


In [ ]:
from blocksnet.analysis.accessibility import area_accessibility

area_acc_blocks = read_cache('area_accessibility.geojson')
if area_acc_blocks is None:
    area_acc = pd.DataFrame({
        'area_accessibility_drive': area_accessibility(acc_mx_drive, buildings_blocks),
        'area_accessibility_intermodal': area_accessibility(acc_mx_intermodal, buildings_blocks),
    })
    area_acc_blocks = blocks[['geometry']].join(area_acc)
    write_cache(area_acc_blocks.assign(block_id=area_acc_blocks.index), 'area_accessibility.geojson')
else:
    area_acc = area_acc_blocks.drop(columns='geometry')
area_acc_blocks = area_acc_blocks.to_crs(crs)

area_acc_blocks.plot('area_accessibility_intermodal', legend=True).set_axis_off()


## 9. Сводные показатели

Собираем все показатели в единый `blocks_metrics.geojson` — он используется на этапе 2 (интерактивная карта и дашборд).


In [ ]:
blocks_metrics = read_cache('blocks_metrics.geojson')
if blocks_metrics is None:
    blocks_metrics = blocks.copy()
    blocks_metrics = blocks_metrics.join(buildings_blocks[['population']])
    blocks_metrics = blocks_metrics.join(prov_df[['demand', 'capacity', 'provision_strong', 'provision_weak']])
    blocks_metrics = blocks_metrics.join(density_df[['fsi', 'gsi', 'mxi', 'l', 'osr']])
    blocks_metrics = blocks_metrics.join(spacematrix_df[['cluster', 'morphotype']].rename(columns={'morphotype': 'spacematrix_morphotype'}))
    blocks_metrics = blocks_metrics.join(strelka_df[['morphotype']].rename(columns={'morphotype': 'strelka_morphotype'}))
    blocks_metrics = blocks_metrics.join(area_acc)
    write_cache(blocks_metrics.assign(block_id=blocks_metrics.index), 'blocks_metrics.geojson')
blocks_metrics = blocks_metrics.to_crs(crs)

blocks_metrics[['population', 'provision_strong', 'area_accessibility_intermodal', 'fsi', 'gsi', 'l', 'mxi', 'spacematrix_morphotype']].head(10)


## 10. Проверка результата

Проверяем, что все файлы кэша на месте и показатели согласованы между собой.


In [ ]:
expected_files = [
    'boundary.geojson',
    'water.geojson',
    'roads.geojson',
    'railways.geojson',
    'blocks_raw.geojson',
    'functional_zones.geojson',
    'blocks_land_use.geojson',
    'saint_petersburg.zip',
    'graph_drive.pickle',
    'graph_intermodal.pickle',
    'acc_mx_drive.csv',
    'acc_mx_intermodal.csv',
    'buildings.geojson',
    'buildings_blocks.geojson',
    'schools.geojson',
    'schools_blocks.geojson',
    'provision_school.geojson',
    'density_indicators.geojson',
    'density_indicators.csv',
    'morphotypes_spacematrix.geojson',
    'morphotypes_spacematrix.csv',
    'morphotypes_spacematrix_clusters.csv',
    'morphotypes_strelka.geojson',
    'morphotypes_strelka.csv',
    'area_accessibility.geojson',
    'blocks_metrics.geojson',
]

missing = [name for name in expected_files if not (DATA_DIR / name).exists()]
assert not missing, f'Не найдены файлы кэша: {missing}'

assert len(blocks) > 0
assert len(blocks_metrics) == len(blocks)
assert 'population' in blocks_metrics.columns
assert acc_mx_drive.shape == (len(blocks), len(blocks))
assert acc_mx_intermodal.shape == (len(blocks), len(blocks))
assert int(acc_mx_drive.isna().sum().sum()) == 0
assert int(acc_mx_intermodal.isna().sum().sum()) == 0
assert list(acc_mx_drive.columns) == list(blocks.index)
assert blocks_metrics.index.equals(blocks.index)

print(f'Кварталов: {len(blocks)}')
print(f'Население: {blocks_metrics.population.sum():,.0f} чел.')
print(f'Средняя обеспеченность школами: {blocks_metrics.provision_strong.mean():.2f}')
print(f'Средняя доступность по площади (ОТ): {blocks_metrics.area_accessibility_intermodal.mean():.1f} мин')
print()
print(blocks_metrics[['population', 'provision_strong', 'area_accessibility_intermodal', 'fsi', 'gsi', 'l', 'osr']].describe().round(2))
print()
print(f'Все {len(expected_files)} файлов кэша на месте')


## 11. Повторный запуск, кэш и дальнейшие шаги

### Состав кэша `examples/data/blocksnet/`

| Файл | Содержимое |
|---|---|
| `boundary.geojson` | граница района |
| `water.geojson`, `roads.geojson`, `railways.geojson` | слои OSM |
| `blocks_raw.geojson` | нарезанные кварталы |
| `functional_zones.geojson` | функциональные зоны OSM |
| `blocks_land_use.geojson` | кварталы с землепользованием |
| `saint_petersburg.zip` | архив BlocksNet с графами (~405 МБ) |
| `graph_drive.pickle`, `graph_intermodal.pickle` | графы (авто / ОТ+пешком) |
| `acc_mx_drive.csv`, `acc_mx_intermodal.csv` | матрицы времени в пути (мин) |
| `buildings.geojson` | здания OSM |
| `buildings_blocks.geojson` | здания, агрегированные по кварталам |
| `schools.geojson`, `schools_blocks.geojson` | школы и их агрегация |
| `provision_school.geojson` | обеспеченность школами |
| `density_indicators.geojson`, `density_indicators.csv` | показатели плотности |
| `morphotypes_spacematrix.geojson`, `morphotypes_spacematrix.csv`, `morphotypes_spacematrix_clusters.csv` | морфотипы Spacematrix |
| `morphotypes_strelka.geojson`, `morphotypes_strelka.csv` | морфотипы Strelka |
| `area_accessibility.geojson` | доступность по площади |
| `blocks_metrics.geojson` | сводная таблица показателей |

### Повторный запуск

- повторный запуск читает данные из кэша, ничего не скачивая заново;
- чтобы пересчитать отдельный шаг — удалите его файл из кэша и запустите ячейку снова;
- чтобы пересчитать всё — удалите папку `examples/data/blocksnet/` целиком;
- GeoJSON хранится в WGS84 (EPSG:4326), вычисления идут в UTM (метры) — при загрузке из кэша данные автоматически перепроецируются.

### Дальнейшие шаги (этап 2)

`blocks_metrics.geojson` и `area_accessibility.geojson` — исходные данные для интерактивной карты и дашборда (отдельный Python-файл, HTML-страница).
